# Ingesta

## Imports

In [13]:
import requests
import datetime
import pandas as pd
import sqlite3
import json
import os

## Transformación + Carga en Raw 

In [2]:
# Configuración de variables globales y conexión a DB

In [3]:
URL = "https://world.openbeautyfacts.org/cgi/search.pl"
NUM_BATCHES = 10
TOTAL_PAGES = 1000
PAGES_PER_BATCH = TOTAL_PAGES//NUM_BATCHES
DB_NAME = "openbeauty"
os.makedirs("data", exist_ok=True)
DB_PATH = f"data/{DB_NAME}.db"

TABLE_NAME = "raw_products"

conn = sqlite3.connect(DB_PATH)


In [4]:
# Datos seleccionados del JSON para usare para el proyecto

In [5]:
def extract_columns(product_dict):
    keys = [
        "_id",
        "code",
        "rev",
        "update_key",
    
        "brands",
        "product_name",
        "product_type",
    
        "countries",
        "countries_tags",
        "countries_hierarchy",
    
        "categories_hierarchy",
        "categories",
    
        "product_quantity",
        "product_quantity_unit",
        "quantity",
    
        "ingredients_n",
        "known_ingredients_n",
        "unknown_ingredients_n",
    
        "completeness",
    
        "scans_n",
        "unique_scans_n",
        "popularity_tags",
    
        "created_t",
        "last_modified_t",
        "last_updated_t",
    
        "creator",
        "last_editor",
        "last_modified_by",
    
        "data_quality_tags"
    ]
    filtered_data = [{k: item.get(k) for k in keys} for item in product_dict]
    return filtered_data

In [6]:
# Función para hacer la petición get a la api, retornado la lista batch_data del numero de paginas correspondiente al batch
# El lote se hará desde initial_page hasta end_page

In [7]:
def load_batch_data(initial_page, end_page):
    batch_data = []

    for n_page in range(initial_page, end_page):
        params = {"search_terms":"","page":n_page, "json":1}
        response = requests.get(URL, params=params)

        if response.status_code == 200:
            data = response.json()
            filtered_data = extract_columns(data['products'])
            # Reconstruye la lista de diccionarios agregando "page":n_page, para control
            filtered_data = [dict(item, **{"page":n_page}) for item in filtered_data] 
            batch_data = batch_data + filtered_data
        
    return batch_data
        

In [8]:
# Función para insertar en base de datos en crudo (raw) los datos resultantes de la función load_batch_data(initial_page, end_page)

In [9]:
def insert_raw_data(batch_data, batch, conn):
    try:
        product_df = pd.DataFrame(batch_data)
        product_df["batch_id"] = batch + 1
        product_df['dtinserted'] = datetime.datetime.now()
        product_df = product_df.map(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
        product_df.to_sql(TABLE_NAME, conn, if_exists="append", index=False)

        print("Data inserted")
        
    except Exception as e:
        print(f"Exception occurs: {e}")
    
        

In [10]:
# Main

In [11]:
def main():
    for batch in range(NUM_BATCHES):
        
        print(f"Processing batch #{batch+1}...")
    
        initial_page = (batch*PAGES_PER_BATCH)+1
        end_page = initial_page + PAGES_PER_BATCH
    
        print(f"Processing page {initial_page} to page {end_page}...")
        batch_data = load_batch_data(initial_page, end_page)
        
        print(f"Inserting to {DB_NAME} database...")
        insert_raw_data(batch_data, batch, conn)
    
        

In [12]:
main()

Processing batch #1...
Processing page 1 to page 101...
Inserting to openbeauty database...
Data inserted
Processing batch #2...
Processing page 101 to page 201...
Inserting to openbeauty database...
Data inserted
Processing batch #3...
Processing page 201 to page 301...
Inserting to openbeauty database...
Data inserted
Processing batch #4...
Processing page 301 to page 401...
Inserting to openbeauty database...
Data inserted
Processing batch #5...
Processing page 401 to page 501...
Inserting to openbeauty database...
Data inserted
Processing batch #6...
Processing page 501 to page 601...
Inserting to openbeauty database...
Data inserted
Processing batch #7...
Processing page 601 to page 701...
Inserting to openbeauty database...
Data inserted
Processing batch #8...
Processing page 701 to page 801...
Inserting to openbeauty database...
Data inserted
Processing batch #9...
Processing page 801 to page 901...
Inserting to openbeauty database...
Data inserted
Processing batch #10...
Proces